# Recurrent Neural Networks in PyTorch

This notebook uses an LSTM for sequence classification on a synthetic time-series dataset. The goal is to focus on recurrent modeling concepts without needing a large external dataset.

## Learning goals

By the end of this notebook, you should be able to:
- represent data as `(batch, sequence, features)`,
- build an `nn.LSTM` classifier,
- train and evaluate a sequence model, and
- interpret learning curves for recurrent networks.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

torch.manual_seed(21)
np.random.seed(21)
random.seed(21)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Create a synthetic sequence dataset

Each sequence is a noisy sine wave. Class 0 uses a lower frequency than class 1, so the network must learn temporal patterns rather than single-point features.

In [ ]:
def make_sequence_dataset(n_samples=2000, seq_len=40):
    X = []
    y = []
    t = np.linspace(0, 1, seq_len)

    for _ in range(n_samples):
        label = np.random.randint(0, 2)
        freq = 2 if label == 0 else 6
        phase = np.random.uniform(0, np.pi)
        signal = np.sin(2 * np.pi * freq * t + phase)
        noise = np.random.normal(scale=0.2, size=seq_len)
        sequence = signal + noise
        X.append(sequence[:, None])
        y.append(label)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

X, y = make_sequence_dataset()
print('Sequence tensor shape:', X.shape)
print('Labels shape:', y.shape)

## Visualize a few sequences

Plotting examples makes the classification task more intuitive.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5), sharex=True, sharey=True)
for ax, seq, label in zip(axes.ravel(), X[:6], y[:6]):
    ax.plot(seq.squeeze())
    ax.set_title(f'Class {label}')
plt.tight_layout()

## Train-validation-test split

Recurrent models expect tensors with three dimensions: batch size, sequence length, and number of features per time step.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=21, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=21, stratify=y_temp)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=64,
    shuffle=True,
)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=128)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=128)

xb, yb = next(iter(train_loader))
print('Mini-batch shape:', xb.shape)

## Define an LSTM classifier

We will use the final hidden state as the summary representation for the whole sequence. Setting `bidirectional=True` lets the network read the sequence in both directions.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, bidirectional=True, num_classes=2):
        super().__init__()
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=0.0,
        )
        direction_factor = 2 if bidirectional else 1
        self.classifier = nn.Linear(hidden_size * direction_factor, num_classes)

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)
        if self.bidirectional:
            final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            final_hidden = hidden[-1]
        return self.classifier(final_hidden)

model = LSTMClassifier().to(device)
model

## Loss, optimizer, and training helpers

Gradient clipping is included because recurrent models can be sensitive to unstable gradients.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(model, loader, criterion, optimizer=None, clip=1.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
                optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += X_batch.size(0)

    return total_loss / total_examples, total_correct / total_examples

## Train the LSTM

On this synthetic task, the model should learn the frequency pattern quickly.

In [ ]:
num_epochs = 12
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"train_acc={train_acc:.3f} | val_acc={val_acc:.3f}"
    )

## Plot training history

These curves make it easier to diagnose convergence and overfitting.

In [ ]:
epochs = np.arange(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history['train_loss'], marker='o', label='Train loss')
axes[0].plot(epochs, history['val_loss'], marker='o', label='Validation loss')
axes[0].set_title('LSTM loss curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], marker='o', label='Train accuracy')
axes[1].plot(epochs, history['val_acc'], marker='o', label='Validation accuracy')
axes[1].set_title('LSTM accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()

## Evaluate on the test split

We now estimate generalization performance on unseen sequences.

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, criterion)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.3f}')

all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(device)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

print(classification_report(all_targets, all_preds))

## Wrap-up

This notebook showed how PyTorch recurrent layers operate on sequence tensors and how an LSTM can solve a sequence classification problem. The same structure extends to text classification, sequence tagging, and more complex encoder-decoder models.